In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

IMG_SIZE = 224
MODEL_PATH = "trained_model/paddy_doctor.keras"
ENCODER_PATH = "trained_model/encoders.pkl"

def load_resources():
    if not os.path.exists(MODEL_PATH) or not os.path.exists(ENCODER_PATH):
        raise FileNotFoundError("Model or Encoder file not found.")

    model = tf.keras.models.load_model(MODEL_PATH)
    with open(ENCODER_PATH, 'rb') as f:
        encoders = pickle.load(f)
    
    print("Resources loaded successfully.")
    return model, encoders

def predict_paddy(model, encoders, img_path):
    
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = image.img_to_array(img)
    img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)
    
    disease_idx = np.argmax(preds[0][0])
    disease_name = encoders['disease_encoder'].inverse_transform([disease_idx])[0]
    
    variety_idx = np.argmax(preds[1][0])
    variety_name = encoders['variety_encoder'].inverse_transform([variety_idx])[0]
    
    max_age = encoders.get('max_age', 100)
    actual_age = int(preds[2][0][0] * max_age)

    return {
        "disease": disease_name,
        "variety": variety_name,
        "age": actual_age,
        "confidence": float(np.max(preds[0][0])) * 100
    }

In [2]:
paddy_model, paddy_encoders = load_resources()

⏳ Loading Model and Encoders...
✅ Resources loaded successfully.


C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
test_path = "100006.jpg"
    
if os.path.exists(test_path):
    res = predict_paddy(paddy_model, paddy_encoders, test_path)
    print("\n🌾 Prediction:")
    print(f"Result: {res['disease']} ({res['confidence']:.2f}%)")
    print(f"Variety: {res['variety']} | Age: {res['age']} days")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

🌾 Prediction:
Result: blast (80.50%)
Variety: ADT45 | Age: 65 days
